
# NB-B — Trigger (SDK) — ClearML Community

Ce notebook enregistre un **TriggerScheduler** côté ClearML Server :
- À chaque **publish** d'un Dataset `image-feedback` du projet, **portant le tag `auto-feedback`**,
  on **enqueue** la `retrain_pipeline` sur la queue `default`.

> ⚠️ Tu dois avoir **créé/exécuté une première fois** la pipeline (NB-C) pour récupérer son `TASK_ID` ou son nom exact.


In [2]:
!pip -q install clearml



[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: python3 -m pip install --upgrade pip


In [3]:
%env CLEARML_WEB_HOST=https://app.clear.ml/
%env CLEARML_API_HOST=https://api.clear.ml
%env CLEARML_FILES_HOST=https://files.clear.ml
# tp3
%env CLEARML_API_ACCESS_KEY=VES7TTVL4QDF36XPHCPXLAV1KDIZY4
%env CLEARML_API_SECRET_KEY=ygijnEsVvuMODXfwaXay-EIcKZbQVReuv18Bd9O5qskRWuHsAbn_V0w1b5waRQbgaNE

env: CLEARML_WEB_HOST=https://app.clear.ml/
env: CLEARML_API_HOST=https://api.clear.ml
env: CLEARML_FILES_HOST=https://files.clear.ml
env: CLEARML_API_ACCESS_KEY=VES7TTVL4QDF36XPHCPXLAV1KDIZY4
env: CLEARML_API_SECRET_KEY=ygijnEsVvuMODXfwaXay-EIcKZbQVReuv18Bd9O5qskRWuHsAbn_V0w1b5waRQbgaNE


In [ ]:
# NB-B — Watcher "latest by date" (no status filter)
# Déclenche la pipeline dès qu'un NOUVEL ID de dataset (par date) apparaît,
# sans filtrer par statut (draft/final).


import time, traceback
from pathlib import Path
from clearml import Task, Dataset

# -------- CONFIG --------
PROJECT = "Image-Classifier/Gradio-Retrain-Demo-IABD"       # Projet du DATASET (comme affiché dans l'UI)
DATASET_NAME = "image-feedback"                        # Nom EXACT du dataset
PIPELINE_TASK_ID = "33714fe795ea49a8adffdfb55770e1b1"  # Run "template" clonable de la pipeline
QUEUE = "controller"                                      # Queue ClearML pour exécuter la pipeline
POLL_SECS = 15
# Use a writable local directory instead of /content, which is only valid in Colab.
STATE_FILE = Path.home() / ".clearml" / ".last_dataset_id"     # Mémoire dernier ID vu
STATE_FILE.parent.mkdir(parents=True, exist_ok=True)
# ------------------------

ctrl = Task.init(project_name=PROJECT, task_name="dataset_watcher_latest", task_type=Task.TaskTypes.controller)
print("Watcher task:", ctrl.id)
print(f"Watching latest dataset name='{DATASET_NAME}' in project='{PROJECT}' | pipeline template={PIPELINE_TASK_ID} | queue='{QUEUE}'")

def _read_last_id():
    try:
        return STATE_FILE.read_text().strip() if STATE_FILE.exists() else ""
    except Exception:
        return ""

def _write_last_id(ds_id: str):
    try:
        STATE_FILE.write_text(ds_id or "")
    except Exception:
        print("⚠️ cannot write state file:", traceback.format_exc())

def _list_matching(dataset_name: str, dataset_project: str):
    try:
        all_entries = Dataset.list_datasets() or []
        return [d for d in all_entries if d.get("name")==dataset_name and d.get("project")==dataset_project]
    except Exception:
        print("❌ list_datasets failed:\n", traceback.format_exc())
        return []

def _latest_by_date(matches):
    """Renvoie la version la plus récente par champ 'created' (AUCUN filtre de statut)."""
    if not matches:
        return None
    return sorted(matches, key=lambda x: x.get("created",""), reverse=True)[0]

def _clone_enqueue_pipeline(template_task_id: str, queue_name: str, run_name_suffix: str = ""):
    from clearml import Task as _Task

    src = _Task.get_task(task_id=template_task_id)
    name = f"{src.name}_auto_{int(time.time())}{run_name_suffix}"

    new_task = _Task.clone(source_task=src, name=name)

    try:
        new_task.set_project(project_name=PROJECT)
    except:
        pass

    _Task.enqueue(
        task=new_task.id,
        queue_name=queue_name
    )

    return new_task.id



last_seen = _read_last_id()
print("Last seen dataset id (from state):", last_seen or "<none>")

print(f"⏳ Start polling every {POLL_SECS}s ... (Ctrl+C to stop)")
while True:
    try:
        matches = _list_matching(DATASET_NAME, PROJECT)

        # Affichage debug
        print("\n🔎 Matching datasets:")
        if not matches:
            print("  ❌ None")
        else:
            for d in sorted(matches, key=lambda x: x.get("created","")):
                print(f"  - id={d['id']} | created={d.get('created')} | status={d.get('status')}")

        latest = _latest_by_date(matches)
        if latest is None:
            print("[poll] No versions yet.")
        else:
            ds_id = latest["id"]
            print(f"[poll] Latest by date: id={ds_id} | created={latest.get('created')} | status={latest.get('status')}")
            if ds_id and ds_id != last_seen:
                print(f"🎯 NEW LATEST DATASET DETECTED: {ds_id} (prev={last_seen or 'none'})")
                run_id = _clone_enqueue_pipeline(PIPELINE_TASK_ID, QUEUE, run_name_suffix=f"__ds_{ds_id[:8]}")
                print(f"✅ Enqueued pipeline run: {run_id} on queue '{QUEUE}'")
                last_seen = ds_id
                _write_last_id(last_seen)
            else:
                print("[poll] Nothing new.")

    except KeyboardInterrupt:
        print("⛔ Stopped by user.")
        break
    except Exception:
        print("❌ watcher loop error:\n", traceback.format_exc())

    time.sleep(POLL_SECS)


ClearML Task: overwriting (reusing) task id=7be74fb738e6489f82e7631a9bc98ee6
ClearML results page: https://app.clear.ml/projects/dc7beb331d174ba3a0cfc2eed86fac20/tasks/7be74fb738e6489f82e7631a9bc98ee6/output/log
2026-10-23 23:28:42,322 - clearml.Task - INFO - No repository found, storing script code instead
2026-10-23 23:28:42,324 - clearml.resource_monitor - WARNING - Could not fetch GPU stats: NVML Shared Library Not Found
Watcher task: 7be74fb738e6489f82e7631a9bc98ee6
Watching latest dataset name='image-feedback' in project='Image-Classifier/Gradio-Retrain-Demo-IABD' | pipeline template=33714fe795ea49a8adffdfb55770e1b1 | queue='controller'
Last seen dataset id (from state): <none>
⏳ Start polling every 15s ... (Ctrl+C to stop)

🔎 Matching datasets:
  ❌ None
[poll] No versions yet.
ClearML Monitor: GPU monitoring failed getting GPU reading, switching off GPU monitoring
